In [3]:
2+2

4

In [4]:
import pandas as pd

In [6]:
import pandas as pd
from datetime import datetime, timedelta, timezone
from binance.client import Client
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import hmean


date_end = None
period = 370
hour = 10 

ticker = 'SOLUSDC'
if date_end is None:
    date_end = datetime.now(timezone.utc)
else:
    date_end = date_end
date_start = date_end - timedelta(days=period)


intervalo_horas = Client.KLINE_INTERVAL_1HOUR

# Inicializar el cliente con tus credenciales
api_key = 'cAGsQPkvI5sxkMt1qTJY7ancjALp1EeabYyF8GltqJ8KuOKvgfZH7aKzYHyQc9X8'
api_secret = 'qkj0jJxLcbzEvpyJvYfww5jzYkv8ExYayhT7mXGbFfeWwT0bnH8VrolE4tNr1WO0'

client = Client(api_key, api_secret)

# Obtener datos históricos
start_timestamp = int(date_start.timestamp() * 1000)
end_timestamp = int(date_end.timestamp() * 1000)

klines = client.get_historical_klines(ticker, intervalo_horas, str(start_timestamp), str(end_timestamp))

if not klines:
    raise ValueError("No se obtuvieron datos históricos para la criptomoneda y el período especificados.")

data = []

def timestamp_to_utc(ts):
    return datetime.fromtimestamp(ts / 1000, tz=timezone.utc)

for kline in klines:
    data.append({
        'time': timestamp_to_utc(float(kline[0])),
        'open': float(kline[1]),
        'high': float(kline[2]),
        'low': float(kline[3]),
        'close': float(kline[4]),
        'volume': float(kline[5]),
    })

aux = pd.DataFrame(data)

In [7]:
aux

,time,open,high,low,close,volume
0,2025-02-10 16:00:00+00:00,203.15,204.43,202.12,203.24,10782.287
1,2025-02-10 17:00:00+00:00,203.26,204.13,202.13,203.62,10086.926
2,2025-02-10 18:00:00+00:00,203.64,203.77,202.25,203.28,6448.564
3,2025-02-10 19:00:00+00:00,203.27,203.46,201.03,201.41,9009.425
4,2025-02-10 20:00:00+00:00,201.42,203.00,201.03,201.16,11689.607
...,...,...,...,...,...,...
8875,2026-02-15 11:00:00+00:00,89.51,89.79,88.96,89.13,9194.002
8876,2026-02-15 12:00:00+00:00,89.14,89.60,87.26,87.28,37633.502
8877,2026-02-15 13:00:00+00:00,87.29,88.34,86.77,86.91,61809.001
8878,2026-02-15 14:00:00+00:00,86.92,87.58,86.61,87.32,101022.429


# Visualización de mi saldo

In [14]:
from binance.client import Client
import time

client = Client(api_key, api_secret)

# 🔥 Sincronizar tiempo correctamente
server_time = client.get_server_time()
server_timestamp = server_time["serverTime"]

local_timestamp = int(time.time() * 1000)

# Calcular diferencia
timestamp_offset = server_timestamp - local_timestamp

# Aplicar al cliente
client.timestamp_offset = timestamp_offset

# Ahora sí debería funcionar
account = client.get_account()

for balance in account["balances"]:
    free = float(balance["free"])
    locked = float(balance["locked"])
    
    if free > 0 or locked > 0:
        print(f"Activo: {balance['asset']}")
        print(f"  Disponible: {free}")
        print(f"  Bloqueado: {locked}")
        print("-" * 20)


Activo: BNB
  Disponible: 2.35e-06
  Bloqueado: 0.0
--------------------
Activo: USDT
  Disponible: 0.08827684
  Bloqueado: 0.0
--------------------
Activo: CVC
  Disponible: 0.601
  Bloqueado: 0.0
--------------------
Activo: DATA
  Disponible: 0.0659
  Bloqueado: 0.0
--------------------
Activo: USDC
  Disponible: 137.97755433
  Bloqueado: 0.0
--------------------
Activo: DOGE
  Disponible: 0.48
  Bloqueado: 0.0
--------------------
Activo: VITE
  Disponible: 0.0975
  Bloqueado: 0.0
--------------------
Activo: FTT
  Disponible: 0.006
  Bloqueado: 0.0
--------------------
Activo: COTI
  Disponible: 0.178
  Bloqueado: 0.0
--------------------
Activo: SOL
  Disponible: 0.00033972
  Bloqueado: 0.0
--------------------
Activo: SUSHI
  Disponible: 0.053
  Bloqueado: 0.0
--------------------
Activo: DIA
  Disponible: 0.0265
  Bloqueado: 0.0
--------------------
Activo: UNI
  Disponible: 0.00227
  Bloqueado: 0.0
--------------------
Activo: AVAX
  Disponible: 0.00024
  Bloqueado: 0.0
------

In [17]:
usdc_balance = next(
    (item for item in account['balances'] if item['asset'] == 'USDC'),
    None
)

print(usdc_balance)


{'asset': 'USDC', 'free': '137.97755433', 'locked': '0.00000000'}


# Order de compra

In [ ]:
from binance.client import Client
from binance.exceptions import BinanceAPIException
import time



client = Client(api_key, api_secret)

# 🔥 Sincronizar reloj para evitar error -1021
server_time = client.get_server_time()["serverTime"]
local_time = int(time.time() * 1000)
client.timestamp_offset = server_time - local_time

try:
    # 🔹 Obtener balance USDC disponible
    usdc_balance = client.get_asset_balance(asset="USDC")
    free_usdc = float(usdc_balance["free"])

    if free_usdc <= 0:
        print("No tienes USDC disponible para comprar.")
    else:
        print(f"USDC disponible: {free_usdc:.2f}")

        # 🔹 Crear orden MARKET usando TODO el USDC
        order = client.create_order(
            symbol="SOLUSDC",
            side="BUY",
            type="MARKET",
            quoteOrderQty=np.round(free_usdc * 0.995,2)  # deja margen para fees
        )

        print("✅ Orden ejecutada:")
        print(order)

except BinanceAPIException as e:
    print("❌ Error de API:", e)
except Exception as e:
    print("❌ Otro error:", e)


USDC disponible: 137.43
✅ Orden ejecutada:
{'symbol': 'SOLUSDC', 'orderId': 3331109685, 'orderListId': -1, 'clientOrderId': 'x-HNA2TXFJe65f7421d3a347d97ef57d', 'transactTime': 1771177287714, 'price': '0.00000000', 'origQty': '1.56700000', 'executedQty': '1.56700000', 'origQuoteOrderQty': '136.74000000', 'cummulativeQuoteQty': '136.68941000', 'status': 'FILLED', 'timeInForce': 'GTC', 'type': 'MARKET', 'side': 'BUY', 'workingTime': 1771177287714, 'fills': [{'price': '87.23000000', 'qty': '1.56700000', 'commission': '0.00148865', 'commissionAsset': 'SOL', 'tradeId': 145425340}], 'selfTradePreventionMode': 'EXPIRE_MAKER'}


In [47]:
from binance.client import Client
from binance.exceptions import BinanceAPIException
import numpy as np
import time



client = Client(api_key, api_secret)

# 🔹 Sincronizar reloj
server_time = client.get_server_time()["serverTime"]
local_time = int(time.time() * 1000)
client.timestamp_offset = server_time - local_time

# 🔹 Función para comprar todo el saldo disponible de la moneda de cotización
def buy_with_full_quote(symbol, fee_margin=0.995):
    """
    Ejecuta orden MARKET comprando con todo el saldo disponible de la moneda de cotización.
    
    Args:
        symbol (str): Par de trading, por ejemplo "SOLUSDC"
        fee_margin (float): Margen para dejar parte del saldo y cubrir fees (por defecto 0.995)
    
    Returns:
        float: Precio medio de ejecución de la compra (approx)
        dict: Detalles completos de la orden
    """
    try:
        # Determinar la moneda de cotización (la parte después de XXX)
        quote_asset = symbol[-4:]  # asumiendo par tipo XXXUSDC

        # Obtener saldo disponible
        balance = client.get_asset_balance(asset=quote_asset)
        free_amount = float(balance['free'])

        if free_amount <= 0:
            print(f"No tienes {quote_asset} disponible para comprar.")
            return None, None

        # Ajustar cantidad a usar según fee_margin
        amount_to_use = round(free_amount * fee_margin, 2)

        # Crear orden MARKET
        order = client.create_order(
            symbol=symbol,
            side="BUY",
            type="MARKET",
            quoteOrderQty=amount_to_use
        )

        # Calcular precio medio de ejecución
        fills = order.get('fills', [])
        if fills:
            total_cost = sum(float(f['price']) * float(f['qty']) for f in fills)
            total_qty = sum(float(f['qty']) for f in fills)
            avg_price = total_cost / total_qty if total_qty > 0 else 0
        else:
            avg_price = 0

        print(f"✅ Orden MARKET ejecutada en {symbol}")
        print(f"Cantidad gastada: {amount_to_use} {quote_asset}, precio medio: {avg_price:.4f}")
        return avg_price, order

    except BinanceAPIException as e:
        print("❌ Error de API:", e)
        return None, None
    except Exception as e:
        print("❌ Otro error:", e)
        return None, None

# 🔹 Ejemplo de uso
symbol = "SOLUSDC"
avg_price, order_details = buy_with_full_quote(symbol)


✅ Orden MARKET ejecutada en SOLUSDC
Cantidad gastada: 134.63 USDC, precio medio: 86.4200


# Orden de venta

In [9]:
avg_price = 86.4200

In [10]:
len(str(avg_price).split('.')[-1])

2

In [15]:
from binance.client import Client
from binance.exceptions import BinanceAPIException
import numpy as np
import time
import threading


client = Client(api_key, api_secret)

# 🔹 Sincronizar reloj
server_time = client.get_server_time()["serverTime"]
local_time = int(time.time() * 1000)
client.timestamp_offset = server_time - local_time

# 🔹 Función para crear orden de venta LIMIT
def create_sell_order(symbol, sell_price):
    try:
        # Obtener saldo disponible del activo base
        base_asset = symbol[:-4]  # asumiendo par tipo XXXUSDC
        balance = client.get_asset_balance(asset=base_asset)
        qty = float(balance['free'])

        if qty <= 0:
            print(f"No tienes {base_asset} disponible para vender.")
            return None

        # Ajustar cantidad según stepSize y minNotional
        symbol_info = client.get_symbol_info(symbol)
        step_size = None
        min_notional = None

        for f in symbol_info['filters']:
            if f['filterType'] == 'LOT_SIZE':
                step_size = float(f['stepSize'])
            elif f['filterType'] == 'NOTIONAL':
                min_notional = float(f['minNotional'])

        if step_size is None or min_notional is None:
            raise ValueError("No se pudo obtener stepSize o minNotional del par")

        qty_to_sell = np.floor(qty / step_size) * step_size

        if qty_to_sell * sell_price < min_notional:
            raise ValueError(f"Valor total de la orden ({qty_to_sell*sell_price:.2f}) menor que minNotional ({min_notional})")

        order = client.create_order(
            symbol=symbol,
            side="SELL",
            type="LIMIT",
            quantity=qty_to_sell,
            price=str(sell_price),
            timeInForce="GTC"
        )

        print(f"✅ Orden de venta creada para {symbol} a {sell_price}:")
        print(order)
        return order

    except BinanceAPIException as e:
        print("❌ Error de API al crear orden de venta:", e)
    except Exception as e:
        print("❌ Otro error al crear orden de venta:", e)

# 🔹 Función para cancelar orden de venta abierta
def cancel_sell_order(symbol):
    try:
        # Obtener todas las órdenes abiertas del par
        open_orders = client.get_open_orders(symbol=symbol)
        if not open_orders:
            print(f"No hay órdenes abiertas para {symbol}.")
            return None

        # Cancelar la primera orden encontrada
        order_id = open_orders[0]['orderId']
        canceled_order = client.cancel_order(symbol=symbol, orderId=order_id)
        print(f"✅ Orden cancelada para {symbol}:")
        print(canceled_order)
        return canceled_order

    except BinanceAPIException as e:
        print("❌ Error de API al cancelar orden:", e)
    except Exception as e:
        print("❌ Otro error al cancelar orden:", e)

# 🔹 Función de supervisión con timer (Stop-Loss manual)
active_timers = {}  # global

def manual_stop_loss(symbol, take_profit_price, stop_price, stop_limit_price, check_interval=2):
    take_profit_order = create_sell_order(symbol, take_profit_price)
    if not take_profit_order:
        return

    def check_price():
        try:
            ticker = client.get_symbol_ticker(symbol=symbol)
            current_price = float(ticker['price'])

            if current_price <= stop_price:
                print(f"⚠️ Precio cayó por debajo de stop_price ({stop_price}). Activando stop-loss...")
                cancel_sell_order(symbol)
                create_sell_order(symbol, stop_limit_price)
                # Cancelamos timer ya que no necesitamos más comprobaciones
                active_timers.pop(symbol, None)
            else:
                # Reprogramar la siguiente comprobación y guardar referencia
                t = threading.Timer(check_interval, check_price)
                active_timers[symbol] = t
                t.start()

        except Exception as e:
            print("❌ Error:", e)
            t = threading.Timer(5, check_price)
            active_timers[symbol] = t
            t.start()

    # 🔹 Primera ejecución
    t = threading.Timer(check_interval, check_price)
    active_timers[symbol] = t
    t.start()



# 🔹 Ejemplo de uso:
symbol = "SOLUSDC"
n = len(str(avg_price).split('.')[-1])
take_profit_price = round(avg_price * 1.015, n)
stop_price = round(avg_price * 0.95, n)
stop_limit_price = round(avg_price * 0.945, n)

manual_stop_loss(symbol, take_profit_price, stop_price, stop_limit_price)

✅ Orden de venta creada para SOLUSDC a 87.72:
{'symbol': 'SOLUSDC', 'orderId': 3331624559, 'orderListId': -1, 'clientOrderId': 'x-HNA2TXFJce63a5ab5e359964dfd4a4', 'transactTime': 1771185396062, 'price': '87.72000000', 'origQty': '1.55500000', 'executedQty': '0.00000000', 'origQuoteOrderQty': '0.00000000', 'cummulativeQuoteQty': '0.00000000', 'status': 'NEW', 'timeInForce': 'GTC', 'type': 'LIMIT', 'side': 'SELL', 'workingTime': 1771185396062, 'fills': [], 'selfTradePreventionMode': 'EXPIRE_MAKER'}


In [22]:
if symbol in active_timers:
    timer = active_timers[symbol]
    print(timer.is_alive())  # True si sigue en ejecución
else:
    print("No hay timer activo para este símbolo")


True


❌ Error: HTTPSConnectionPool(host='api.binance.com', port=443): Read timed out. (read timeout=10)
❌ Otro error: HTTPSConnectionPool(host='api.binance.com', port=443): Read timed out. (read timeout=10)


In [1]:
from binance.client import Client
from binance.exceptions import BinanceAPIException
import numpy as np
import time

api_key = 'cAGsQPkvI5sxkMt1qTJY7ancjALp1EeabYyF8GltqJ8KuOKvgfZH7aKzYHyQc9X8'
api_secret = 'qkj0jJxLcbzEvpyJvYfww5jzYkv8ExYayhT7mXGbFfeWwT0bnH8VrolE4tNr1WO0'

client = Client(api_key, api_secret)

# 🔹 Sincronizar reloj
server_time = client.get_server_time()["serverTime"]
local_time = int(time.time() * 1000)
client.timestamp_offset = server_time - local_time

# 🔹 Función para crear orden de venta LIMIT
def create_sell_order(symbol, sell_price):
    try:
        # Obtener saldo disponible del activo base
        base_asset = symbol[:-4]  # asumiendo par tipo XXXUSDC
        balance = client.get_asset_balance(asset=base_asset)
        qty = float(balance['free'])

        if qty <= 0:
            print(f"No tienes {base_asset} disponible para vender.")
            return None

        # Ajustar cantidad según stepSize y minNotional
        symbol_info = client.get_symbol_info(symbol)
        step_size = None
        min_notional = None

        for f in symbol_info['filters']:
            if f['filterType'] == 'LOT_SIZE':
                step_size = float(f['stepSize'])
            elif f['filterType'] == 'NOTIONAL':
                min_notional = float(f['minNotional'])

        if step_size is None or min_notional is None:
            raise ValueError("No se pudo obtener stepSize o minNotional del par")

        qty_to_sell = np.floor(qty / step_size) * step_size

        if qty_to_sell * sell_price < min_notional:
            raise ValueError(f"Valor total de la orden ({qty_to_sell*sell_price:.2f}) menor que minNotional ({min_notional})")

        order = client.create_order(
            symbol=symbol,
            side="SELL",
            type="LIMIT",
            quantity=qty_to_sell,
            price=str(sell_price),
            timeInForce="GTC"
        )

        print(f"✅ Orden de venta creada para {symbol} a {sell_price}:")
        print(order)
        return order

    except BinanceAPIException as e:
        print("❌ Error de API al crear orden de venta:", e)
    except Exception as e:
        print("❌ Otro error al crear orden de venta:", e)

# 🔹 Función para cancelar orden de venta abierta
def cancel_sell_order(symbol):
    try:
        # Obtener todas las órdenes abiertas del par
        open_orders = client.get_open_orders(symbol=symbol)
        if not open_orders:
            print(f"No hay órdenes abiertas para {symbol}.")
            return None

        # Cancelar la primera orden encontrada
        order_id = open_orders[0]['orderId']
        canceled_order = client.cancel_order(symbol=symbol, orderId=order_id)
        print(f"✅ Orden cancelada para {symbol}:")
        print(canceled_order)
        return canceled_order

    except BinanceAPIException as e:
        print("❌ Error de API al cancelar orden:", e)
    except Exception as e:
        print("❌ Otro error al cancelar orden:", e)

# 🔹 Función de bucle de supervisión para Stop-Loss manual
def manual_stop_loss(symbol, take_profit_price, stop_price, stop_limit_price, check_interval=2):


    # 2️⃣ Bucle de supervisión
    while True:
        try:
            ticker = client.get_symbol_ticker(symbol=symbol)
            current_price = float(ticker['price'])
            # print(f"Precio actual: {current_price}")  # opcional

            if current_price <= stop_price:
                print(f"⚠️ Precio cayó por debajo de stop_price ({stop_price}). Activando stop-loss...")

                # Cancelar take-profit
                cancel_sell_order(symbol)

                create_sell_order(symbol, stop_limit_price)
                break

            time.sleep(check_interval)

        except BinanceAPIException as e:
            print("❌ Error de API:", e)
        except Exception as e:
            print("❌ Otro error:", e)

# 🔹 Ejemplo de uso:
symbol = "SOLUSDC"
n = len(str(avg_price).split('.')[-1])
take_profit_price = round(avg_price*1.015,n)
stop_price = round(avg_price*0.95,n)
stop_limit_price = round(avg_price*0.945,n)

manual_stop_loss(symbol, take_profit_price, stop_price, stop_limit_price)


NameError: name 'avg_price' is not defined